In [1]:
!python -m pip install pandas sentence-transformers faiss-cpu


[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

C:\Users\Valentina J K\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = {
    "title": [
        "Circe",
        "Atomic Habits",
        "Harry Potter",
        "The Poppy War",
        "The Alchemist",
        "Dune",
        "The Night Circus",
        "Educated",
        "The Hobbit",
        "Thinking Fast and Slow"
    ],
    
    "genre": [
        "Fantasy",
        "Self Help",
        "Fantasy",
        "Fantasy",
        "Fiction",
        "Science Fiction",
        "Fantasy",
        "Memoir",
        "Fantasy",
        "Psychology"
    ],
    
    "description": [
        "Greek mythological story of a powerful witch",
        "Guide to building good habits and breaking bad ones",
        "A young wizard discovering magic",
        "War and magic story with a powerful heroine",
        "A shepherd searching for treasure and meaning",
        "Political conflict on a desert planet",
        "A magical competition between illusionists",
        "A woman escaping a survivalist family",
        "A hobbit going on an unexpected adventure",
        "Explains how humans think and make decisions"
    ]
}

df = pd.DataFrame(data)

df

,title,genre,description
0,Circe,Fantasy,Greek mythological story of a powerful witch
1,Atomic Habits,Self Help,Guide to building good habits and breaking bad...
2,Harry Potter,Fantasy,A young wizard discovering magic
3,The Poppy War,Fantasy,War and magic story with a powerful heroine
4,The Alchemist,Fiction,A shepherd searching for treasure and meaning
5,Dune,Science Fiction,Political conflict on a desert planet
6,The Night Circus,Fantasy,A magical competition between illusionists
7,Educated,Memoir,A woman escaping a survivalist family
8,The Hobbit,Fantasy,A hobbit going on an unexpected adventure
9,Thinking Fast and Slow,Psychology,Explains how humans think and make decisions


In [4]:
import pandas as pd

data = {
    "title": [
        "Circe",
        "Atomic Habits",
        "Harry Potter",
        "The Poppy War",
        "The Alchemist",
        "Dune",
        "The Night Circus",
        "Educated",
        "The Hobbit",
        "Thinking Fast and Slow"
    ],
    
    "genre": [
        "Fantasy",
        "Self Help",
        "Fantasy",
        "Fantasy",
        "Fiction",
        "Science Fiction",
        "Fantasy",
        "Memoir",
        "Fantasy",
        "Psychology"
    ],
    
    "description": [
        "Greek mythological story of a powerful witch",
        "Guide to building good habits and breaking bad ones",
        "A young wizard discovering magic",
        "War and magic story with a powerful heroine",
        "A shepherd searching for treasure and meaning",
        "Political conflict on a desert planet",
        "A magical competition between illusionists",
        "A woman escaping a survivalist family",
        "A hobbit going on an unexpected adventure",
        "Explains how humans think and make decisions"
    ]
}

df = pd.DataFrame(data)

print(df)

                    title            genre  \
0                   Circe          Fantasy   
1           Atomic Habits        Self Help   
2            Harry Potter          Fantasy   
3           The Poppy War          Fantasy   
4           The Alchemist          Fiction   
5                    Dune  Science Fiction   
6        The Night Circus          Fantasy   
7                Educated           Memoir   
8              The Hobbit          Fantasy   
9  Thinking Fast and Slow       Psychology   

                                         description  
0       Greek mythological story of a powerful witch  
1  Guide to building good habits and breaking bad...  
2                   A young wizard discovering magic  
3        War and magic story with a powerful heroine  
4      A shepherd searching for treasure and meaning  
5              Political conflict on a desert planet  
6         A magical competition between illusionists  
7              A woman escaping a survivalist family 

In [5]:
df["combined_text"] = df["title"] + " " + df["genre"] + " " + df["description"]

df[["title","combined_text"]]

,title,combined_text
0,Circe,Circe Fantasy Greek mythological story of a po...
1,Atomic Habits,Atomic Habits Self Help Guide to building good...
2,Harry Potter,Harry Potter Fantasy A young wizard discoverin...
3,The Poppy War,The Poppy War Fantasy War and magic story with...
4,The Alchemist,The Alchemist Fiction A shepherd searching for...
5,Dune,Dune Science Fiction Political conflict on a d...
6,The Night Circus,The Night Circus Fantasy A magical competition...
7,Educated,Educated Memoir A woman escaping a survivalist...
8,The Hobbit,The Hobbit Fantasy A hobbit going on an unexpe...
9,Thinking Fast and Slow,Thinking Fast and Slow Psychology Explains how...


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████████████████| 103/103 [00:00<00:00, 4265.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embeddings = model.encode(df["combined_text"].tolist())

print(embeddings.shape)

(10, 384)


In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Vector database created!")

Vector database created!


In [9]:
def recommend_books(query, top_k=5):
    
    # convert query to embedding
    query_vector = model.encode([query])
    
    # search the vector database
    distances, indices = index.search(np.array(query_vector), top_k)
    
    # get recommended books
    results = df.iloc[indices[0]][["title","genre"]]
    
    return results

In [10]:
recommend_books("dark fantasy magic")

,title,genre
2,Harry Potter,Fantasy
6,The Night Circus,Fantasy
3,The Poppy War,Fantasy
0,Circe,Fantasy
8,The Hobbit,Fantasy


In [11]:
recommend_books("wizard magic school")

,title,genre
2,Harry Potter,Fantasy
6,The Night Circus,Fantasy
3,The Poppy War,Fantasy
0,Circe,Fantasy
8,The Hobbit,Fantasy


In [12]:
recommend_books("space politics desert")

,title,genre
5,Dune,Science Fiction
6,The Night Circus,Fantasy
3,The Poppy War,Fantasy
7,Educated,Memoir
8,The Hobbit,Fantasy


In [13]:
recommend_books("self improvement habits")

,title,genre
1,Atomic Habits,Self Help
9,Thinking Fast and Slow,Psychology
2,Harry Potter,Fantasy
6,The Night Circus,Fantasy
4,The Alchemist,Fiction


In [14]:
def retrieve_context(query, top_k=3):
    
    query_vector = model.encode([query])
    
    distances, indices = index.search(np.array(query_vector), top_k)
    
    results = df.iloc[indices[0]]
    
    context = ""
    
    for _, row in results.iterrows():
        context += f"Book: {row['title']}\nGenre: {row['genre']}\nDescription: {row['description']}\n\n"
    
    return context

In [15]:
print(retrieve_context("dark fantasy magic"))

Book: Harry Potter
Genre: Fantasy
Description: A young wizard discovering magic

Book: The Night Circus
Genre: Fantasy
Description: A magical competition between illusionists

Book: The Poppy War
Genre: Fantasy
Description: War and magic story with a powerful heroine




In [16]:
print(retrieve_context("magic wizard school"))

Book: Harry Potter
Genre: Fantasy
Description: A young wizard discovering magic

Book: The Night Circus
Genre: Fantasy
Description: A magical competition between illusionists

Book: The Poppy War
Genre: Fantasy
Description: War and magic story with a powerful heroine




In [17]:
!python -m pip install transformers


[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

Loading weights: 100%|█████████████████████████████████| 148/148 [00:00<00:00, 3069.51it/s]


In [19]:
def ask_sherlock(query):
    
    context = retrieve_context(query)
    
    prompt = f"""
You are an intelligent book recommendation assistant.

User query: {query}

Relevant books:
{context}

Explain why these books may be good recommendations.
"""
    
    response = generator(prompt, max_length=200, num_return_sequences=1)
    
    return response[0]['generated_text']

In [20]:
print(ask_sherlock("dark fantasy magic"))

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an intelligent book recommendation assistant.

User query: dark fantasy magic

Relevant books:
Book: Harry Potter
Genre: Fantasy
Description: A young wizard discovering magic

Book: The Night Circus
Genre: Fantasy
Description: A magical competition between illusionists

Book: The Poppy War
Genre: Fantasy
Description: War and magic story with a powerful heroine



Explain why these books may be good recommendations.

User query: books of the good old days

Relevant books:

Book: The Lost World of Harry Potter

Genre: English

Description: A family drama with two adults

Book: The Ravenclaw Temple

Genre: English

Description: A magical ritual involving wizards and witches

Book: The Secret of the Dark Tower

Genre: English

Description: A magical journey through time

Book: The Dark Lord's Curse

Genre: English

Description: A journey through the Middle Ages to the end of time


Book: A Story of the Witches

Genre: English

Description: A magical tale of witches and wizards

Bo

In [21]:
def ask_sherlock(query):

    query_vector = model.encode([query])
    distances, indices = index.search(np.array(query_vector), 3)

    results = df.iloc[indices[0]]

    response = f"\n  Query: {query}\n\n Recommended Books:\n"

    for _, row in results.iterrows():

        response += f"""
Title: {row['title']}
Genre: {row['genre']}
Description: {row['description']}

Why recommended:
This book matches your interest in "{query}" because it involves {row['description']} and belongs to the {row['genre']} genre.

"""

    return response

In [22]:
print(ask_sherlock("dark fantasy magic"))


  Query: dark fantasy magic

 Recommended Books:

Title: Harry Potter
Genre: Fantasy
Description: A young wizard discovering magic

Why recommended:
This book matches your interest in "dark fantasy magic" because it involves A young wizard discovering magic and belongs to the Fantasy genre.


Title: The Night Circus
Genre: Fantasy
Description: A magical competition between illusionists

Why recommended:
This book matches your interest in "dark fantasy magic" because it involves A magical competition between illusionists and belongs to the Fantasy genre.


Title: The Poppy War
Genre: Fantasy
Description: War and magic story with a powerful heroine

Why recommended:
This book matches your interest in "dark fantasy magic" because it involves War and magic story with a powerful heroine and belongs to the Fantasy genre.


